In [ ]:
# Reference: https://www.kaggle.com/code/jhoward/is-it-a-bird-creating-a-model-from-your-own-data

In [3]:
!uv add fastai fastbook fastdownload fastcore duckduckgo-search pillow

Resolved 215 packages in 8.57s                                       
Prepared 48 packages in 2m 50s                                           
Installed 120 packages in 1.37s                             
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + beartype==0.22.9
 + blis==1.3.3
 + catalogue==2.0.10
 + click==8.4.2
 + cloudpathlib==0.24.0
 + cloudpickle==3.1.2
 + confection==1.3.3
 + contourpy==1.3.3
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.6.0
 + cuda-toolkit==13.0.3.0
 + cycler==0.12.1
 + cymem==2.0.13
 + datasets==5.0.1
 + dill==0.4.1
 + duckduckgo-search==8.1.1
 + fastai==2.8.8
 + fastbook==0.0.29
 + fastcore==2.1.19
 + fastdownload==0.0.7
 + fastprogress==1.1.6
 + fasttransform==0.0.2
 + filelock==3.32.2
 + fonttools==4.63.0
 + frozenlist==1.8.0
 + fsspec==2026.6.0
 + graphviz==0.21
 + hf-xet==1.6.0
 + httpcore2==2.9.1
 + httptools==0.8.0
 + httpx2==2.9.1
 + huggingface-hub==1.26.0
 + ipython-genutil

In [ ]:
# !pip install -Uqq fastai 
# U is for upgrade so that we always have the latest version of fastai

In [9]:
import time
import json
from pathlib import Path
from PIL import Image

import fastbook
fastbook.setup_book()

from fastai.vision.all import *
from fastdownload import download_url # to help download urls
from fastcore.all import *
from duckduckgo_search import DDGS

# Fastai provides us with a lot of helpful modules to do taks and they usually start with 'fast'

In [5]:
def search_images(keywords, max_images=200): return L(DDGS().images(keywords, max_results=max_images)).itemgot('image')
import time, json

In [10]:
#`search_images` depends on duckduckgo.com, which doesn't always return correct responses.

urls = search_images('bird photos', max_images=1)
urls[0]

/tmp/ipykernel_13488/616225101.py:1: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  def search_images(keywords, max_images=200): return L(DDGS().images(keywords, max_results=max_images)).itemgot('image')


RatelimitException: https://duckduckgo.com/i.js?o=json&q=bird+photos&l=us-en&vqd=4-115142509036086875169432515889989043843&p=1&f=%2C%2C%2C%2C%2C 403 Ratelimit

In [ ]:
from fastdownload import download_url
dest = 'bird.jpg'
download_url(urls[0], dest, show_progress=False)

from fastai.vision.all import *
im = Image.open(dest)
# if the last line of a cell has something that can be shown, it will be shown 
im.to_thumb(256,256) # t convert into a thumbnail

In [ ]:
download_url(search_images('forest photos', max_images=1)[0], 'forest.jpg', show_progress=False)
Image.open('forest.jpg').to_thumb(256,256)

In [ ]:
searches = 'forest','bird'
path = Path('bird_or_not')

for o in searches:
    dest = (path/o)
    dest.mkdir(exist_ok=True, parents=True)
    download_images(dest, urls=search_images(f'{o} photo')) # download images is from fastai where you just provide a list of images and it does that in parallel
    time.sleep(5)
    resize_images(path/o, max_size=400, dest=path/o) # resize image to 400 so that it is faster, otherwise most of the time would be wasted in jut opening the image

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink) # delete broken images
len(failed)

In [ ]:
# How do I get my data into the model?
# blocks: What kinds of inputs do we have? FOr us, our input is an ImageBlock and the output is a CategoryBlock. That's enough for fastai to know what kind of model to build
# get_items: What are we training on? get_image_files is a function here which returns a list of all the image files in a path based on extension
# splitter: Create a validation set aside for testing; fastai won't let you proceed without this
# get_y: How do we know it is the correct label of a photo aka how do we know it is a birfd or a forrest? parent_label here is a function that returns the parent folder of a path
# item_tfms: Most computer vision architectures need all your inputs as you train to be the same size. item_transforms are all the bits that will run on every item and we resize each of them to 192x192 pixels and we squish the image

# https://docs.fast.ai/data.block.html
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=get_image_files, 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)

# pytorch iterates through 'dataloaders' to grab a bunch of data at a time, usually using a GPU
# A data loader will feed the training algorithm with a "bunch" of your images at a bunch
# This bunch is called a "batch" or a "mini batch"

dls.show_batch(max_n=6) # Show an example of a batch of data that will be passed into the model

In [ ]:
# A learner is something that combines a model (the neural netowrk we are training) adnd the data we use to train it with
# Here, the data is the data loaders ovject (dls)
# The model is the actual neural network function

# https://timm.fast.ai/
# Pytorch image models is the largest collection of computer vision models in the world and fast.ai integrates this

# The resnet family model is good enough for most things
# This model is already trained to recognise over 1 million images of over 1000 different types (the ImageNet database) and they then made those weights available
# By default fast.ai downloads the weights to so that we don't start with a neural network that can't do anything
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

# fine_tune takes the downloaded weights and adjusts them in a really carefully controlled way to just teach the model the differences between your data set and what it was orginally trained for

In [ ]:
is_bird,_,probs = learn.predict(PILImage.create('bird.jpg'))
print(f"This is a: {is_bird}.")
print(f"Probability it's a bird: {probs[0]:.4f}")